# Fetal Ultrasound: Dataset Setup and Integrity Checks

**Project:** Explainable Transfer Learning Framework for Fetal Ultrasound Standard Plane Classification  
**Dataset:** FETAL_PLANES_DB (Burgos-Artizzu et al., 2020)

This Google Colab notebook prepares the ultrasound dataset on a new Google account. It reuses the **original, verified patient-disjoint manifest**, class mapping and audit record, downloading the original image archive only when needed.

If Google Drive contains duplicate `splits` or `results` folders, this notebook can locate the original records by their verified content, without renaming, deleting or recreating those folders. Only the verified record copies are staged temporarily in Colab. No original split record is edited.

| Partition | Images | Distinct patient IDs |
|:--|--:|--:|
| Training | 5,818 | 716 |
| Validation | 1,311 | 180 |
| Held-out TEST | 5,271 | 896 |
| **Total** | **12,400** | **1,792** |

Run the cells in order. A CPU runtime is sufficient. This notebook does not train models, select checkpoints, or repeat the completed held-out TEST evaluation.


## 1. Mount Google Drive

This step mounts Drive and defines the ultrasound workspace. It does **not** create the `splits` or `results` folders. Duplicate folders already in Google Drive are left untouched.


In [1]:
from google.colab import drive
from pathlib import Path

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/ICTIU_Final_Projects')
PROJECT = ROOT / 'ultrasound'
WORK = Path('/content/ictiu_data/ultrasound')

if not PROJECT.is_dir():
    PROJECT.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

ARCHIVE = PROJECT / 'archives' / 'FETAL_PLANES_ZENODO.zip'
METADATA_FILE = PROJECT / 'metadata' / 'FETAL_PLANES_DB_data.csv'

print('Ultrasound project:', PROJECT)
print('Temporary workspace:', WORK)
print('Existing Drive splits/results folders were not created, renamed or modified.')


Mounted at /content/drive
Ultrasound project: /content/drive/MyDrive/ICTIU_Final_Projects/ultrasound
Temporary workspace: /content/ictiu_data/ultrasound
Existing Drive splits/results folders were not created, renamed or modified.


In [2]:
# Resolve verified records, including when Google Drive has duplicate folder names.
# The Google Drive API is used READ-ONLY if the normal mounted path is ambiguous.
import hashlib
import json
from pathlib import Path

EXPECTED_MANIFEST_SHA256 = 'ef6fec1a70e80a0b7ed34c990ebac2918ddc8a81dcef300aa244ce6ab1a5e1e0'
EXPECTED_MAPPING_SHA256 = '26ec8f0a94d29d3756f36c52f02ddaad0d953403c0cd3d3903a243547d693428'

source_paths = {
    'manifest_candidate.csv': PROJECT / 'splits' / 'manifest_candidate.csv',
    'class_mapping.json': PROJECT / 'splits' / 'class_mapping.json',
    'audit.json': PROJECT / 'results' / 'audit.json',
}


def verified_record(name, data):
    if name == 'manifest_candidate.csv':
        return hashlib.sha256(data).hexdigest() == EXPECTED_MANIFEST_SHA256
    if name == 'class_mapping.json':
        return hashlib.sha256(data).hexdigest() == EXPECTED_MAPPING_SHA256
    if name == 'audit.json':
        try:
            record = json.loads(data.decode('utf-8'))
            return (record.get('manifest_sha256') == EXPECTED_MANIFEST_SHA256
                    and record.get('status') == 'PASS_candidate_split_checks')
        except (UnicodeError, ValueError, AttributeError):
            return False
    return False


resolved = {}
for name, path in source_paths.items():
    try:
        content = path.read_bytes() if path.is_file() else None
    except OSError:
        content = None
    if content is not None and verified_record(name, content):
        resolved[name] = content

missing = [name for name in source_paths if name not in resolved]
if missing:
    print('Drive folder names are ambiguous or records are not at their usual paths.')
    print('Searching Drive for the exact ORIGINAL record contents...')
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    import io

    auth.authenticate_user()  # May ask for the Google account once.
    drive_api = build('drive', 'v3', cache_discovery=False)

    def find_original_file(name):
        # Searches only this filename, never arbitrary Drive content.
        query = f"name = '{name}' and trashed = false"
        page_token = None
        checked = 0
        while True:
            page = drive_api.files().list(
                q=query,
                fields='nextPageToken,incompleteSearch,files(id,mimeType,name)',
                pageSize=100,
                pageToken=page_token,
            ).execute()
            for item in page.get('files', []):
                if item.get('mimeType') == 'application/vnd.google-apps.folder':
                    continue
                if item.get('mimeType', '').startswith('application/vnd.google-apps.'):
                    continue
                checked += 1
                if checked > 100:
                    raise RuntimeError(f'Too many same-name Drive files for {name}; organize backups first.')
                buffer = io.BytesIO()
                downloader = MediaIoBaseDownload(buffer, drive_api.files().get_media(fileId=item['id']))
                done = False
                while not done:
                    _, done = downloader.next_chunk(num_retries=3)
                candidate = buffer.getvalue()
                if verified_record(name, candidate):
                    return candidate
            page_token = page.get('nextPageToken')
            if not page_token:
                break
        return None

    for name in missing:
        data = find_original_file(name)
        if data is None:
            raise FileNotFoundError(
                f'Could not find a VERIFIED {name} in the signed-in Google Drive. '
                'Check that Colab is connected to the account with the original copied records. '
                'Do not generate a replacement split.'
            )
        resolved[name] = data

# Use unique LOCAL copies, so duplicate Drive folder names cannot affect later cells.
# These are temporary copies only; no Drive file is overwritten or moved.
RECORD_DIR = WORK / 'verified_records'
RECORD_DIR.mkdir(parents=True, exist_ok=True)
for name, content in resolved.items():
    output = RECORD_DIR / name
    output.write_bytes(content)
    if not verified_record(name, output.read_bytes()):
        raise RuntimeError(f'Failed to preserve verified original record: {name}')

MANIFEST_FILE = RECORD_DIR / 'manifest_candidate.csv'
MAPPING_FILE = RECORD_DIR / 'class_mapping.json'
AUDIT_FILE = RECORD_DIR / 'audit.json'

print('PASS: Original manifest, class mapping, and audit loaded and verified.')
print('Temporary verified inputs:', RECORD_DIR)
print('No duplicate Drive folders changed; no split regenerated.')


PASS: Original manifest, class mapping, and audit loaded and verified.
Temporary verified inputs: /content/ictiu_data/ultrasound/verified_records
No duplicate Drive folders changed; no split regenerated.


## 2. Verify the saved ultrasound partitions

The previous cell selects the original verified manifest, class mapping and audit, even if Drive contains duplicate folder names. If the normal Drive paths are ambiguous, it may request read-only Google Drive API access. The exact manifest and class-mapping SHA-256 values and patient-separated counts are checked below.

The held-out TEST partition is checked using **manifest metadata only**. No TEST image pixels or model predictions are read.


In [3]:
import hashlib
import json
from pathlib import PurePosixPath
import pandas as pd

CLASS_IDS = {
    'Fetal abdomen': 0,
    'Fetal brain': 1,
    'Fetal femur': 2,
    'Fetal thorax': 3,
    'Maternal cervix': 4,
    'Other': 5,
}
EXPECTED_IMAGES = {'train': 5818, 'validation': 1311, 'test': 5271}
EXPECTED_PATIENTS = {'train': 716, 'validation': 180, 'test': 896}
EXPECTED_MANIFEST_SHA256 = 'ef6fec1a70e80a0b7ed34c990ebac2918ddc8a81dcef300aa244ce6ab1a5e1e0'
EXPECTED_MAPPING_SHA256 = '26ec8f0a94d29d3756f36c52f02ddaad0d953403c0cd3d3903a243547d693428'


def file_digest(path, algorithm='sha256'):
    digest = hashlib.new(algorithm)
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


for original in (MANIFEST_FILE, MAPPING_FILE, AUDIT_FILE):
    if not original.is_file():
        raise FileNotFoundError(
            f'Restore the original audited Drive record: {original}. '
            'Do not generate a new split or replace a lost checksum.'
        )

manifest_hash = file_digest(MANIFEST_FILE)
mapping_hash = file_digest(MAPPING_FILE)
if manifest_hash != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Saved ultrasound manifest SHA-256 differs from the original verified record.')
if mapping_hash != EXPECTED_MAPPING_SHA256:
    raise RuntimeError('Saved ultrasound class mapping SHA-256 differs from the original verified record.')

mapping = json.loads(MAPPING_FILE.read_text(encoding='utf-8'))
audit = json.loads(AUDIT_FILE.read_text(encoding='utf-8'))
if mapping != CLASS_IDS:
    raise RuntimeError('Unexpected class IDs. The original six-class mapping must be retained.')
if audit.get('manifest_sha256') != manifest_hash or audit.get('status') != 'PASS_candidate_split_checks':
    raise RuntimeError('The original audit and manifest do not agree.')

manifest = pd.read_csv(
    MANIFEST_FILE,
    dtype={'stem': str, 'relative_path': str, 'group_id': str, 'sha256': str},
    keep_default_na=False,
)
needed = {'stem', 'relative_path', 'group_id', 'class_name', 'class_id',
          'official_train', 'split', 'sha256'}
if not needed.issubset(manifest.columns):
    raise RuntimeError(f'Missing original manifest fields: {sorted(needed - set(manifest.columns))}')
for col in ('stem', 'relative_path', 'group_id', 'class_name', 'sha256', 'split'):
    manifest[col] = manifest[col].astype(str).str.strip()
if (manifest[list(('stem', 'relative_path', 'group_id', 'class_name', 'sha256', 'split'))] == '').any().any():
    raise RuntimeError('The manifest contains a blank identifier, class, path, or hash.')
manifest['class_id'] = pd.to_numeric(manifest['class_id'], errors='raise').astype(int)
manifest['official_train'] = pd.to_numeric(manifest['official_train'], errors='raise').astype(int)

if len(manifest) != 12400 or manifest['group_id'].nunique() != 1792:
    raise RuntimeError('Unexpected dataset row count or patient count.')
if set(manifest['split']) != set(EXPECTED_IMAGES) or set(manifest['class_name']) != set(CLASS_IDS):
    raise RuntimeError('Unexpected split labels or plane categories.')
if manifest['split'].value_counts().to_dict() != EXPECTED_IMAGES:
    raise RuntimeError('Image counts differ from the frozen split.')
if manifest.groupby('split')['group_id'].nunique().to_dict() != EXPECTED_PATIENTS:
    raise RuntimeError('Patient counts differ from the frozen split.')
if manifest.groupby('group_id')['split'].nunique().max() != 1:
    raise RuntimeError('A patient appears in more than one partition.')
if manifest['class_id'].ne(manifest['class_name'].map(CLASS_IDS)).any():
    raise RuntimeError('Class indices do not match saved labels.')
if not ((manifest['official_train'] == 0) == (manifest['split'] == 'test')).all():
    raise RuntimeError('The official dataset test allocation is not preserved.')
if manifest['relative_path'].duplicated().any() or manifest['stem'].duplicated().any():
    raise RuntimeError('Duplicate image path or filename stem in saved manifest.')
if not manifest['sha256'].str.fullmatch(r'[a-fA-F0-9]{64}').all():
    raise RuntimeError('Malformed original image SHA-256 record.')
if manifest.groupby('sha256')['split'].nunique().max() != 1:
    raise RuntimeError('Original manifest has exact-duplicate image hashes spanning splits.')
for rel in manifest['relative_path']:
    path = PurePosixPath(rel)
    if path.is_absolute() or '..' in path.parts or '\\' in rel or path.suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
        raise RuntimeError(f'Invalid manifest image path: {rel!r}')
if pd.crosstab(manifest['split'], manifest['class_name']).eq(0).any().any():
    raise RuntimeError('Some class is missing from a partition.')

print('Saved manifest and label map: PASS')
print('Verified manifest SHA-256:', manifest_hash)
print('Dataset image counts:', manifest['split'].value_counts().reindex(['train','validation','test']).to_dict())
print('Distinct patients:', manifest.groupby('split')['group_id'].nunique().reindex(['train','validation','test']).to_dict())
print('Metadata checked for TEST; no TEST image pixels read.')

Saved manifest and label map: PASS
Verified manifest SHA-256: ef6fec1a70e80a0b7ed34c990ebac2918ddc8a81dcef300aa244ce6ab1a5e1e0
Dataset image counts: {'train': 5818, 'validation': 1311, 'test': 5271}
Distinct patients: {'train': 716, 'validation': 180, 'test': 896}
Metadata checked for TEST; no TEST image pixels read.


## 3. Obtain the original Zenodo archive only if missing

Source: [Zenodo FETAL_PLANES_DB, record 3904280](https://zenodo.org/records/3904280), with the originally published MD5 checksum `2a5fcc2cefb789bcc0f6c1f73e0ea43f`. The existing complete archive in Drive is **reused, not downloaded again**. If the archive is absent, this cell downloads the **same original release** into `ultrasound/archives/`, using a `.part` file to avoid mistaking an interrupted download for a finished ZIP. An invalid existing ZIP causes a stop rather than a silent replacement.

This download is approximately 2.1 GB. Colab network availability and Google Drive capacity are not guaranteed. If automatic access fails, download it manually from the Zenodo record and place it in your Drive's `ICTIU_Final_Projects/ultrasound/archives/` folder with the exact name `FETAL_PLANES_ZENODO.zip`, then rerun this section. Never commit the dataset ZIP to GitHub.

In [4]:
from pathlib import Path
import zipfile
import requests
from tqdm.auto import tqdm

ZENODO_URL = 'https://zenodo.org/records/3904280/files/FETAL_PLANES_ZENODO.zip?download=1'
EXPECTED_ARCHIVE_MD5 = '2a5fcc2cefb789bcc0f6c1f73e0ea43f'


def verify_archive(path):
    if not zipfile.is_zipfile(path):
        raise RuntimeError(f'Archive is not a readable ZIP: {path}')
    actual_md5 = file_digest(path, 'md5')
    if actual_md5.lower() != EXPECTED_ARCHIVE_MD5:
        raise RuntimeError(f'Archive MD5 mismatch: {actual_md5}. Stop and inspect the file.')
    return actual_md5


def fetch_original_archive(url, destination):
    destination = Path(destination)
    if not destination.parent.is_dir():
        destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        print('Reusing existing dataset ZIP from Drive.')
        verify_archive(destination)
        return destination

    partial = destination.with_name(destination.name + '.part')
    offset = partial.stat().st_size if partial.exists() else 0
    headers = {'Accept-Encoding': 'identity'}
    if offset:
        headers['Range'] = f'bytes={offset}-'
    print('Original archive missing in Drive. Downloading the Zenodo release...')
    with requests.get(url, stream=True, timeout=(30, 180), headers=headers, allow_redirects=True) as response:
        response.raise_for_status()
        if not response.url.startswith('https://'):
            raise RuntimeError('The dataset request redirected to a non-HTTPS URL.')
        if 'text/html' in response.headers.get('Content-Type', '').lower():
            raise RuntimeError('A web page was returned instead of the dataset ZIP.')
        if response.status_code == 206 and offset:
            if not response.headers.get('Content-Range', '').startswith(f'bytes {offset}-'):
                raise RuntimeError('Invalid resume offset. Keep the .part file for inspection.')
            mode, initial = 'ab', offset
        elif response.status_code == 200:
            # The server ignored Range; restart the partial file instead of appending bad bytes.
            mode, initial = 'wb', 0
        else:
            raise RuntimeError(f'Unexpected HTTP download status: {response.status_code}')
        size_text = response.headers.get('Content-Length', '')
        total = initial + int(size_text) if size_text.isdigit() else None
        with partial.open(mode) as stream, tqdm(
            desc='Zenodo ultrasound archive', total=total, initial=initial,
            unit='B', unit_scale=True
        ) as progress:
            for block in response.iter_content(chunk_size=4 * 1024 * 1024):
                if block:
                    stream.write(block)
                    progress.update(len(block))
    verify_archive(partial)  # Never publish a partial or corrupt ZIP as completed.
    partial.replace(destination)
    return destination


ARCHIVE = fetch_original_archive(ZENODO_URL, ARCHIVE)
print('Archive integrity PASS. Published MD5:', verify_archive(ARCHIVE))
print('Saved archive:', ARCHIVE)

Reusing existing dataset ZIP from Drive.
Archive integrity PASS. Published MD5: 2a5fcc2cefb789bcc0f6c1f73e0ea43f
Saved archive: /content/drive/MyDrive/ICTIU_Final_Projects/ultrasound/archives/FETAL_PLANES_ZENODO.zip


## 4. Restore temporary ultrasound images from the verified archive

The existing manifest determines every restored file path. The default is to restore **training and validation** images only, so the previously evaluated TEST image pixels remain unopened. `RESTORE_SPLITS` may be changed strictly for an authorized archival-recovery need, not to rerun the final TEST evaluation.

Archive entries are checked for unsafe paths, symbolic links and duplicate names. Existing temporary files are reused when they match the archive entries; selected image hashes are verified in the later integrity check.


In [5]:
import shutil
import stat
import zipfile
from pathlib import PurePosixPath
from tqdm.auto import tqdm

RESTORE_SPLITS = ('train', 'validation')
# To recover ALL files for archival purposes only, explicitly use:
# RESTORE_SPLITS = ('train', 'validation', 'test')

if not set(RESTORE_SPLITS).issubset({'train', 'validation', 'test'}):
    raise ValueError('Invalid RESTORE_SPLITS option.')

selected = manifest.loc[manifest['split'].isin(RESTORE_SPLITS), 'relative_path'].tolist()
expected_set = set(selected)
if len(expected_set) != len(selected):
    raise RuntimeError('The manifest contains duplicate selected paths.')


def restore_from_zip(archive, destination_root, relative_paths):
    destination_root = Path(destination_root)
    destination_root.mkdir(parents=True, exist_ok=True)
    safe_root = destination_root.resolve()
    wanted = set(relative_paths)
    if not wanted:
        return 0, 0

    with zipfile.ZipFile(archive) as zf:
        names = {}
        for entry in zf.infolist():
            if entry.filename not in wanted:
                continue
            rel = PurePosixPath(entry.filename)
            if (entry.is_dir() or rel.is_absolute() or '..' in rel.parts or
                    '\\' in entry.filename or stat.S_ISLNK(entry.external_attr >> 16)):
                raise RuntimeError(f'Unsafe ZIP member: {entry.filename!r}')
            if entry.filename in names:
                raise RuntimeError(f'Duplicate ZIP entry: {entry.filename!r}')
            names[entry.filename] = entry

        missing_zip = wanted.difference(names)
        if missing_zip:
            raise RuntimeError(f'Manifest and original archive disagree. Example: {sorted(missing_zip)[0]}')
        if sum(entry.file_size for entry in names.values()) > 20 * 1024**3:
            raise RuntimeError('Unexpected uncompressed archive size.')
        pending = []
        for name in relative_paths:
            entry = names[name]
            target = destination_root / name
            if not target.resolve().is_relative_to(safe_root):
                raise RuntimeError('Unsafe extraction target.')
            if target.is_symlink():
                raise RuntimeError('A target path is a symbolic link.')
            if target.is_file() and target.stat().st_size == entry.file_size:
                continue
            pending.append((entry, target))

        required = sum(entry.file_size for entry, _ in pending)
        if pending and shutil.disk_usage(safe_root).free < required + 512 * 1024**2:
            raise RuntimeError('Colab temporary disk has insufficient free space.')
        for entry, target in tqdm(pending, desc='Restore ultrasound image files'):
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + '.extracting')
            with zf.open(entry) as source, temporary.open('wb') as output:
                shutil.copyfileobj(source, output)
            temporary.replace(target)
    return len(relative_paths), len(pending)


expected, recovered = restore_from_zip(ARCHIVE, WORK, selected)
missing = [relative for relative in selected if not (WORK / relative).is_file()]
if missing:
    raise RuntimeError(f'Files missing after restoration. First missing: {missing[0]}')
print(f'Restore PASS: {expected:,} selected images are present; {recovered:,} newly extracted.')
print('RESTORE_SPLITS:', RESTORE_SPLITS)
print('Model training and evaluation: NOT performed.')

Restore ultrasound image files:   0%|          | 0/7129 [00:00<?, ?it/s]

Restore PASS: 7,129 selected images are present; 7,129 newly extracted.
RESTORE_SPLITS: ('train', 'validation')
Model training and evaluation: NOT performed.


## 5. Recover/check the original metadata CSV, without changing split records

This check compares the dataset's supplied `Image_name`, `Patient_num`, `Plane` and `Train` values with the entire saved manifest. If the separate Drive metadata copy is missing, the CSV can be recovered from the already-verified original ZIP. No patient identifiers or individual image paths are printed. TEST rows are metadata-only.

In [7]:
import io
from pathlib import Path, PurePosixPath
import pandas as pd


def ensure_original_metadata(archive, destination):
    destination = Path(destination)
    if destination.is_file():
        return destination
    with zipfile.ZipFile(archive) as zf:
        found = [info for info in zf.infolist()
                 if PurePosixPath(info.filename).name == 'FETAL_PLANES_DB_data.csv'
                 and not info.is_dir()]
        if len(found) != 1:
            raise RuntimeError(f'Expected one original metadata CSV, found {len(found)}.')
        info = found[0]
        rel = PurePosixPath(info.filename)
        if rel.is_absolute() or '..' in rel.parts or '\\' in info.filename or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError('Unsafe metadata path in original ZIP.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + '.extracting')
        with zf.open(info) as source, temporary.open('wb') as output:
            shutil.copyfileobj(source, output)
        temporary.replace(destination)
    print('Restored the original metadata CSV to Drive (no split files changed).')
    return destination


metadata_file = ensure_original_metadata(ARCHIVE, METADATA_FILE)
original_metadata = pd.read_csv(metadata_file, sep=';')
original_metadata.columns = original_metadata.columns.str.strip()
required_metadata = {'Image_name', 'Patient_num', 'Plane', 'Train'}
if not required_metadata.issubset(original_metadata.columns):
    raise RuntimeError('Original FETAL_PLANES_DB metadata columns do not match the project record.')
if original_metadata[list(required_metadata)].isna().any().any():
    raise RuntimeError('Missing original metadata values.')
original_metadata['stem'] = original_metadata['Image_name'].astype(str).str.strip().map(lambda x: Path(x).stem)
original_metadata['class_name'] = original_metadata['Plane'].astype(str).str.strip()
original_metadata['group_id'] = original_metadata['Patient_num'].astype(str).str.strip()
original_metadata['official_train'] = pd.to_numeric(original_metadata['Train'], errors='raise').astype(int)
if len(original_metadata) != 12400 or original_metadata['stem'].duplicated().any():
    raise RuntimeError('Unexpected metadata row count or duplicate ultrasound image stem.')
comparison = manifest[['stem', 'class_name', 'group_id', 'official_train']].merge(
    original_metadata[['stem', 'class_name', 'group_id', 'official_train']],
    how='outer', on='stem', validate='one_to_one', suffixes=('_saved', '_source'), indicator=True
)
if not comparison['_merge'].eq('both').all():
    raise RuntimeError('Original metadata and frozen manifest list different images.')
for column in ('class_name', 'group_id', 'official_train'):
    if not comparison[column + '_saved'].eq(comparison[column + '_source']).all():
        raise RuntimeError(f'Original metadata differs from frozen manifest: {column}')
print('Original metadata ↔ saved manifest: PASS (12,400 rows).')
print('All original patient-separated split assignments remain unchanged.')

Original metadata ↔ saved manifest: PASS (12,400 rows).
All original patient-separated split assignments remain unchanged.


## 6. TRAIN and VALIDATION image spot checks

The completed ultrasound preparation audit recorded no unreadable images, split-overlapping patient groups or cross-split byte-identical images. This cell performs a deterministic spot check of **one training and one validation image from each of the six classes**. It checks each file's stored SHA-256 and decodes the image without inspecting TEST pixels.

These spot checks do not replace the complete study audit. The output reports aggregate counts, not patient identifiers or raw ultrasound images.


In [8]:
from PIL import Image

sample_rows = pd.concat([
    manifest.loc[manifest['split'].eq(part)].sort_values('relative_path')
            .groupby('class_name', sort=True, group_keys=False).head(1)
    for part in ('train', 'validation')
], ignore_index=True)
if len(sample_rows) != 12:
    raise RuntimeError('Expected one train and one validation image per class.')

for row in sample_rows.itertuples(index=False):
    image_file = WORK / row.relative_path
    if not image_file.is_file():
        raise FileNotFoundError('Run the restoration section before image spot-checking.')
    if file_digest(image_file) != row.sha256.lower():
        raise RuntimeError('Recovered image bytes differ from the original manifest SHA-256.')
    with Image.open(image_file) as image:
        image.load()
        if min(image.size) <= 0:
            raise RuntimeError('Invalid decoded ultrasound image dimensions.')

print('PASS: 12/12 TRAIN/VALIDATION images decoded and matched their stored SHA-256 values.')
print('Frozen manifest:', EXPECTED_MANIFEST_SHA256)
print('Dataset source: https://zenodo.org/records/3904280')
print('No new split, model training, Grad-CAM, or held-out TEST evaluation was performed.')

PASS: 12/12 TRAIN/VALIDATION images decoded and matched their stored SHA-256 values.
Frozen manifest: ef6fec1a70e80a0b7ed34c990ebac2918ddc8a81dcef300aa244ce6ab1a5e1e0
Dataset source: https://zenodo.org/records/3904280
No new split, model training, Grad-CAM, or held-out TEST evaluation was performed.


## Reproducibility and provenance

- **Fixed split evidence:** The verified original patient-separated manifest, class mapping and audit are required. The notebook checks their exact contents and does not regenerate split assignments.
- **Duplicate Google Drive folders:** The notebook can search for verified input records by file ID and stage temporary copies. Original Drive files and folders remain untouched.
- **Dataset:** Burgos-Artizzu, X. P., et al. (2020). *FETAL_PLANES_DB: Common maternal-fetal ultrasound images*. Zenodo. https://doi.org/10.5281/zenodo.3904280.
- **Related publication:** Burgos-Artizzu, X. P., et al. (2020). *Evaluation of deep convolutional neural networks for automatic classification of common maternal fetal ultrasound planes*. Scientific Reports, 10, 10200. https://doi.org/10.1038/s41598-020-67076-5.
- **Artifact handling:** Model checkpoints and raw medical images remain in controlled storage. Do not commit raw dataset archives, credentials or restricted source images to GitHub.
- **Evaluation:** This notebook performs no new training, checkpoint selection, or held-out TEST evaluation. The ultrasound classification task does not constitute clinical diagnosis or external-center validation.

**Execution state:** This revised setup notebook contains no prefilled code outputs. Run it on the intended Colab Google account, then save the actual executed version before committing to GitHub.
